## Lab 19 – Iceberg Tables in Snowflake
**Snowflake Fundamentals 4-day class Lab · © 2026 Innovation In Software Corporation. All rights reserved.**

Topics covered:
1. Creating a role with Iceberg-specific privileges
2. Setting up an External Volume backed by Amazon S3
3. Creating an Iceberg Table with Snowflake as the catalog
4. DML on Iceberg tables — INSERT, UPDATE, DELETE
5. Verifying data files in S3 (Parquet + metadata)
6. Time Travel on Iceberg tables — AT OFFSET, AT TIMESTAMP, BEFORE STATEMENT

### DEMO 1 | Create Role: iceberg_management_role

> **[INSTRUCTOR NOTE]**
> Iceberg tables require `CREATE EXTERNAL VOLUME` at the account level. A dedicated role keeps these powerful privileges isolated from day-to-day roles. Grant this role to the current user after creation.

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE ROLE iceberg_management_role;

GRANT CREATE DATABASE,
      CREATE EXTERNAL VOLUME
ON ACCOUNT
TO ROLE iceberg_management_role;

### DEMO 2 | Create Database and Schema

> **[INSTRUCTOR NOTE]**
> A dedicated database and schema keep Iceberg objects isolated from other lab objects.

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE DATABASE iceberg_demo_db;
CREATE OR REPLACE SCHEMA iceberg_demo_db.iceberg_schema;

### DEMO 3 | Grant Privileges to iceberg_management_role

> **[INSTRUCTOR NOTE]**
> `CREATE ICEBERG TABLE` must be granted at the schema level. USAGE on the database and schema lets the role navigate to the objects it creates.

In [ ]:
%%sql -r dataframe_3
GRANT CREATE ICEBERG TABLE ON SCHEMA iceberg_demo_db.iceberg_schema TO ROLE iceberg_management_role;

GRANT USAGE ON DATABASE iceberg_demo_db TO ROLE iceberg_management_role;
GRANT USAGE ON SCHEMA iceberg_demo_db.iceberg_schema TO ROLE iceberg_management_role;

### DEMO 4 | Create a New S3 Bucket

> **[INSTRUCTOR NOTE]**
> Switch to the AWS Console and create a new S3 bucket in the **same region** as your Snowflake account.
>
> - Navigate to https://aws.amazon.com/console/
> - Find your **AWS Account ID** (e.g. 595239850147)
> - Create a new S3 bucket — e.g. `iceberg-storage-<yourname>`
> - Note the bucket name; it will be used in the next step.

### DEMO 5 | Create External Volume

> **[INSTRUCTOR NOTE]**
> The Python cell builds the `CREATE EXTERNAL VOLUME` DDL from variables, making it easy to customise per student. The role ARN references the IAM role we will create in the next steps using the Snowflake-provided external ID.

In [ ]:
aws_account_id = 595239850147
s3_bucket      = 'iceberg-storage-zahar-h'
role_name      = 'SnowflakeIcebergAccessRole'
ext_volume     = 'ICEBERG_EXTERNAL_VOLUME'
policy_name    = 'InlineSnowflakeIcebergS3'

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

sql = f"""
    CREATE OR REPLACE EXTERNAL VOLUME {ext_volume}
    STORAGE_LOCATIONS = (
        (
          NAME = 'default_s3_loc'
          STORAGE_PROVIDER = 'S3'
          STORAGE_BASE_URL = 's3://{s3_bucket}/'
          STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::{aws_account_id}:role/{role_name}'
        )
    )
    ALLOW_WRITES = TRUE
    COMMENT = 'Volume for Iceberg tables in {s3_bucket}'
    """

print(sql)

session.sql(sql).collect()
print(f"EXTERNAL VOLUME {ext_volume} has been created")

### DEMO 6 | Describe the External Volume

> **[INSTRUCTOR NOTE]**
> `DESC EXTERNAL VOLUME` returns the Snowflake-managed IAM user ARN and the External ID. Both values are required to configure the trust policy of the AWS IAM role in the next step.

In [ ]:
%%sql -r dataframe_6
DESC EXTERNAL VOLUME ICEBERG_EXTERNAL_VOLUME
->> SELECT "property_value" FROM $1 WHERE "property" = 'STORAGE_LOCATION_1';

In [ ]:
%%sql -r dataframe_7
DESC EXTERNAL VOLUME ICEBERG_EXTERNAL_VOLUME
->>
SELECT
    PARSE_JSON("property_value") j
FROM $1
WHERE "property" = 'STORAGE_LOCATION_1'
->>
SELECT
    SPLIT_PART(SPLIT_PART(j:STORAGE_AWS_IAM_USER_ARN::STRING, '::', 2), ':', 1) AS snowflake_aws_account_id,
    j:STORAGE_AWS_EXTERNAL_ID::STRING AS external_id
FROM $1;

In [ ]:
_sql = """
DESC EXTERNAL VOLUME ICEBERG_EXTERNAL_VOLUME
->>
SELECT PARSE_JSON(\"property_value\") j
FROM $1
WHERE \"property\" = 'STORAGE_LOCATION_1'
->>
SELECT
  SPLIT_PART(SPLIT_PART(j:STORAGE_AWS_IAM_USER_ARN::STRING, '::', 2), ':', 1) AS snowflake_aws_account_id,
  j:STORAGE_AWS_EXTERNAL_ID::STRING AS external_id
FROM $1;
"""

row = session.sql(_sql).collect()[0]

sf_acct     = row["SNOWFLAKE_AWS_ACCOUNT_ID"]
external_id = row["EXTERNAL_ID"]

print("Snowflake AWS Account ID:", sf_acct)
print("External ID:", external_id)

### DEMO 7 | Generate IAM Trust and S3 Policy JSON

> **[INSTRUCTOR NOTE]**
> The Python cell builds the two JSON policy documents needed in AWS IAM:
> 1. **Trust policy** — allows Snowflake's AWS account to assume the role, scoped to our External ID.
> 2. **Inline S3 policy** — grants `ListBucket`, `GetObject`, `PutObject`, `DeleteObject` on the specific bucket.
>
> Copy each JSON block into AWS IAM as shown in the instructions below.

In [ ]:
import json

trust_doc = {
  "Version": "2012-10-17",
  "Statement": [{
      "Effect": "Allow",
      "Principal": {"AWS": f"arn:aws:iam::{sf_acct}:root"},
      "Action": "sts:AssumeRole",
      "Condition": {"StringEquals": {"sts:ExternalId": external_id}}
  }]
}

s3_policy = {
  "Version": "2012-10-17",
  "Statement": [
    {"Sid": "BucketLevel", "Effect": "Allow",
     "Action": ["s3:ListBucket", "s3:GetBucketLocation"],
     "Resource": f"arn:aws:s3:::{s3_bucket}"},
    {"Sid": "ObjectLevel", "Effect": "Allow",
     "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject", "s3:DeleteObjectVersion"],
     "Resource": f"arn:aws:s3:::{s3_bucket}/*"}
  ]
}

print(f"=== Create Role: {role_name} ===")
print("=== Trust policy ===")
print(json.dumps(trust_doc, indent=2))
print(f"\n=== Inline S3 policy: {policy_name} ===")
print(json.dumps(s3_policy, indent=2))

### AWS IAM Setup Instructions

In the AWS IAM Console:

1. Go to **Roles** → **Create Role**
2. Choose **Custom trust policy** and paste the Trust Policy JSON from above, then click **Next**
3. Skip "Add permissions" and click **Next**
4. Enter Role Name: `SnowflakeIcebergAccessRole` and click **Create Role**
5. Find the newly created role and click on it
6. Under **Add permissions** → **Create inline policy**
7. Switch to JSON view, paste the Inline S3 Policy JSON from above, click **Next**
8. Enter Policy name: `InlineSnowflakeIcebergS3` and click **Create Policy**

### DEMO 8 | Create Iceberg Table

> **[INSTRUCTOR NOTE]**
> `CATALOG = 'SNOWFLAKE'` means Snowflake manages the Iceberg metadata (no external catalog like AWS Glue required). `BASE_LOCATION` sets the S3 subdirectory under the external volume where data and metadata files are stored.

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE ICEBERG TABLE iceberg_demo_db.iceberg_schema.orders (
    order_id      INT,
    customer_id   INT,
    order_date    DATE,
    total_amount  NUMBER(10,2)
)
CATALOG         = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'ICEBERG_EXTERNAL_VOLUME'
BASE_LOCATION   = 'orders/';

### DEMO 9 | Insert Rows

> **[INSTRUCTOR NOTE]**
> Standard DML works on Snowflake-managed Iceberg tables. Each write creates a new Iceberg snapshot — Parquet data files and JSON metadata files appear in S3 under `orders/`.

In [ ]:
%%sql -r dataframe_11
INSERT INTO iceberg_demo_db.iceberg_schema.orders
    (order_id, customer_id, order_date, total_amount)
VALUES
    (1, 101, '2025-09-17', 150.75),
    (2, 102, '2025-09-18', 200.00);

### DEMO 10 | Verify in S3

> **[INSTRUCTOR NOTE]**
> Open your S3 bucket and look for a new folder structure like:
>
> ```
> orders.<random_suffix>/
>     metadata/   -- Iceberg metadata & snapshots (JSON / Avro)
>     data/       -- Parquet files with your rows
> ```
>
> Refresh S3 (`data/`) after the UPDATE and DELETE below: additional snapshot/manifest files will appear. Old Parquet files are NOT overwritten — Iceberg uses copy-on-write semantics.

In [ ]:
%%sql -r dataframe_12
-- Optional: prove updates/deletes create new Iceberg snapshots
UPDATE iceberg_demo_db.iceberg_schema.orders
SET total_amount = 175.00
WHERE order_id = 1;

DELETE FROM iceberg_demo_db.iceberg_schema.orders
WHERE order_id = 2;

In [ ]:
%%sql -r dataframe_13
SELECT * FROM iceberg_demo_db.iceberg_schema.orders;

### DEMO 11 | Time Travel with Iceberg Tables

> **[INSTRUCTOR NOTE]**
> Snowflake-managed Iceberg tables support Time Travel exactly like native tables. The default retention is 1 day. Show all three syntax variants: `AT OFFSET`, `AT TIMESTAMP`, and `BEFORE STATEMENT`.

In [ ]:
%%sql -r dataframe_14
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN TABLE iceberg_demo_db.iceberg_schema.orders;

In [ ]:
%%sql -r dataframe_15
-- Check the table as of 3 minutes ago
SELECT * FROM iceberg_demo_db.iceberg_schema.orders AT (OFFSET => -3*60);

In [ ]:
%%sql -r dataframe_16
-- Check the table as of a specific point in time
SELECT *
FROM iceberg_demo_db.iceberg_schema.orders AT (
    TIMESTAMP => DATEADD(minute, -4, CURRENT_TIMESTAMP())
);

In [ ]:
%%sql -r dataframe_17
SET id = (
    SELECT query_id
    FROM TABLE(information_schema.query_history_by_session())
    WHERE query_type = 'DELETE'
    ORDER BY start_time DESC
    LIMIT 1
);

In [ ]:
%%sql -r dataframe_18
SELECT *
FROM iceberg_demo_db.iceberg_schema.orders
BEFORE (STATEMENT => $id);

### DEMO CLEANUP

> Drop Snowflake objects and then delete the S3 bucket and IAM role in the AWS Console.

In [ ]:
%%sql -r dataframe_19
-- DROP ROLE            IF EXISTS iceberg_management_role;
-- DROP DATABASE        IF EXISTS iceberg_demo_db;
-- DROP EXTERNAL VOLUME IF EXISTS ICEBERG_EXTERNAL_VOLUME;

-- In AWS Console: delete the S3 bucket and the SnowflakeIcebergAccessRole IAM role